# Goals analysis: what targets are recorded and how do they change?
For the internship presentation, Friday October 23, 2026.

This notebook complements `steps_analysis.ipynb`: it focuses on the three targets and their changes, with a separate floors comparison. It does not repeat steps-goal attainment. Only goals, steps and activity are read. No database writes, CSV exports or predictive models are used. Outputs are aggregates without participant identifiers.

## 1. Inspect every variable and load only required tables
One candidate row is a participant's set of targets for a labelled measurement period. A synthetic example is “participant A, one labelled date/period, steps target 8,000, floors target 10, intensity target 150.” The last number has **unknown units and scope**; it must not be interpreted as minutes or a weekly/daily target without documentation.

`additional_id` is a candidate record identifier; user/group/type fields identify the record context. Date/week/day label the record; start/end/duration describe the reporting period. Steps and floors targets are provisionally counts based on names. SQL types do not establish business units. We inspect all 13 columns, including contextual fields, without displaying real rows.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, inspect, text
from dotenv import dotenv_values
from IPython.display import display

PROJECT=Path.cwd()
url=dotenv_values(PROJECT/'.env').get('DATABASE_URL')
assert url, 'Configure DATABASE_URL privately in the project .env.'
engine=create_engine(url)
tables={}; schema=[]
try:
    inspector=inspect(engine)
    for name in ['goals','steps','activity']:
        for c in inspector.get_columns(name):
            schema.append({'table':name,'column':c['name'],'sql_type':str(c['type'])})
        with engine.connect() as conn:
            tables[name]=pd.read_sql(text('SELECT * FROM '+name),conn)
finally:
    engine.dispose()
del url
for df in tables.values():
    df['calendar_date']=pd.to_datetime(df.calendar_date,errors='coerce')
g=tables['goals'].copy()
targets=['steps_goal','floors_goal','intensity_goal']
display(pd.DataFrame(schema).query("table == 'goals'"))
display(pd.DataFrame({'column':g.columns,'pandas_type':g.dtypes.astype(str).values,
    'missing_rows':g.isna().sum().values,'distinct_nonnull':g.nunique().values}))
print('Goals records:',len(g),'Participants:',g.sdv_user_id.nunique())


## 2. Establish record grain and period quality
Check exact duplicates and candidate keys separately. A participant/date key being unique supports one record per observed date in this snapshot, not complete daily measurement. Cross-table matching later uses participant/date/start/end, preventing different periods from being compared by accident. No duplicate rows are silently removed.

Duration is checked as reported seconds, including end-minus-start consistency. A reporting window cannot establish actual device wear or the time horizon over which a target should be achieved. Week fields are checked against ISO date conventions only as a diagnostic; a mismatch may reflect a different convention.

In [ ]:
key=['sdv_user_id','calendar_date','start_measure_period_sec','end_measure_period_sec']
def valid_key(df,k):
    return not df[k].isna().any(axis=None) and not df.duplicated(k).any()
audit=[]
for name,df in tables.items():
    for label,k in [('record_id',['additional_id']),('participant_date',['sdv_user_id','calendar_date']),('participant_period',key)]:
        audit.append({'table':name,'key':label,'null_key_rows':int(df[k].isna().any(axis=1).sum()),
          'rows_in_duplicate_keys':int(df.duplicated(k,keep=False).sum()),'valid_unique_key':valid_key(df,k)})
display(pd.DataFrame(audit))
print('Exact duplicate goals rows after first:',int(g.duplicated().sum()))
elapsed=g.end_measure_period_sec-g.start_measure_period_sec
display(pd.DataFrame([{'invalid_nonpositive_duration':int(g.duration_measure_period_sec.le(0).sum()),
 'end_not_after_start':int(elapsed.le(0).sum()),
 'duration_matches_end_minus_start':int(g.duration_measure_period_sec.eq(elapsed).sum()),
 'ISO_week_matches':int(g.week_nr.eq(g.calendar_date.dt.isocalendar().week).sum()),
 'ISO_weekday_matches':int(g.week_day.eq(g.calendar_date.dt.dayofweek+1).sum())}]))
display((g.duration_measure_period_sec/3600).value_counts().sort_index().rename_axis('reported_hours').to_frame('records'))
display(g[['week_nr','week_day']].agg(['min','max','nunique']).T)


## 3. Describe targets, including invalid or undefined targets
Show missing, negative, zero and positive counts separately. Zero is an observed value, not a missing value; it may mean no target or a source-specific sentinel. Positive-target distributions describe recorded target settings, not recommended targets. Each record has equal weight, so participants with more observations contribute more.

In [ ]:
quality=[]
for target in targets:
    x=g[target]
    quality.append({'target':target,'missing':int(x.isna().sum()),'negative':int(x.lt(0).sum()),
        'zero':int(x.eq(0).sum()),'positive':int(x.gt(0).sum()),'distinct_nonnull':x.nunique()})
display(pd.DataFrame(quality))
display(g[targets].describe().T)
for target in targets:
    display(g[target].value_counts(dropna=False).sort_index().rename_axis(target).to_frame('records'))
fig,axes=plt.subplots(1,3,figsize=(13,4))
for ax,target in zip(axes,targets):
    g[target].dropna().plot.hist(bins=20,ax=ax)
    ax.set_title(target); ax.set_ylabel('Recorded periods')
    ax.set_xlabel('Recorded target value'+(' (units unconfirmed)' if target=='intensity_goal' else ''))
plt.tight_layout(); plt.show()


## 4. Measure changes within participants
Sort observations by participant, calendar date and period. Compare each target with that participant's previous observed value. The first record has no predecessor; transitions involving a missing target are excluded and counted. We report any recorded-value change, with increases/decreases and positive-to-positive changes separately.

A change across a date gap does not tell us the exact day the target changed. A transition is a pair of adjacent **observations**, not necessarily adjacent days. These comparisons are skipped if the participant/date key is not unique or any ordering key is missing. We do not infer adaptive algorithms, compliance or causality from changes.

In [ ]:
change_summary=[]
if not valid_key(g,['sdv_user_id','calendar_date']) or g[key].isna().any(axis=None):
    print('Changes skipped: resolve ambiguous dates or missing ordering keys first.')
else:
    ordered=g.sort_values(key).copy()
    grouped=ordered.groupby('sdv_user_id',sort=False)
    prev_date=grouped.calendar_date.shift()
    has_previous=prev_date.notna()
    gap=(ordered.calendar_date-prev_date).dt.days
    for target in targets:
        previous=grouped[target].shift()
        comparable=has_previous & previous.notna() & ordered[target].notna()
        delta=ordered[target]-previous
        changed=comparable & delta.ne(0)
        positive_pair=comparable & previous.gt(0) & ordered[target].gt(0)
        change_summary.append({'target':target,'first_observations':int((~has_previous).sum()),
          'comparable_transitions':int(comparable.sum()),
          'missing_value_transitions':int((has_previous & ~comparable).sum()),
          'changed_transitions':int(changed.sum()),'increases':int((comparable & delta.gt(0)).sum()),
          'decreases':int((comparable & delta.lt(0)).sum()),
          'changed_across_date_gap':int((changed & gap.gt(1)).sum()),
          'positive_to_positive_changes':int((positive_pair & delta.ne(0)).sum()),
          'participants_with_change':ordered.loc[changed,'sdv_user_id'].nunique()})
    display(pd.DataFrame(change_summary))
    distinct=grouped[targets].nunique()
    display(distinct.describe().T.rename_axis('distinct_target_values_per_participant'))
    print('Transitions separated by more than one calendar day:',int((has_previous & gap.gt(1)).sum()))


## 5. Validate relationships with steps and activity
Recheck exact-period keys against this load, rather than relying on equal row counts or a previous snapshot. Outer joins count unmatched periods; `validate='one_to_one'` prevents row multiplication. Compare the duplicated steps target with steps and compare contextual fields for both joins. Do not join on target values. Intensity is deliberately not compared with activity durations because its units, weighting and daily/weekly scope are unconfirmed.

In [ ]:
joins={}; join_summary=[]; agreement=[]
for name in ['steps','activity']:
    right=tables[name]
    if not (valid_key(g,key) and valid_key(right,key)):
        print(name+': skipped because period keys are not unique and non-null.')
        continue
    joined=g.merge(right,on=key,how='outer',suffixes=('_goals','_other'),indicator=True,validate='one_to_one')
    joins[name]=joined
    both=joined._merge.eq('both')
    join_summary.append({'table':name,'matched':int(both.sum()),'goals_only':int(joined._merge.eq('left_only').sum()),
                         'other_only':int(joined._merge.eq('right_only').sum())})
    for field in ['additional_id','sdv_group_id','duration_measure_period_sec']+(['steps_goal'] if name=='steps' else []):
        a=joined.loc[both,field+'_goals']; b=joined.loc[both,field+'_other']
        present=a.notna() & b.notna()
        agreement.append({'table':name,'field':field,'both_nonnull':int(present.sum()),
            'equal':int((present & a.eq(b)).sum()),'different':int((present & a.ne(b)).sum()),
            'missing_either':int((~present).sum())})
display(pd.DataFrame(join_summary)); display(pd.DataFrame(agreement))


## 6. Floors achievement: distinguish missing from zero
Use only matched periods whose group and duration agree. A usable goal is strictly positive. Missing floor counts are excluded, while recorded zero floors remain eligible. Negative floors are flagged and excluded. Display exclusions separately; categories can overlap. Meeting the goal means recorded floors ≥ the recorded positive floors goal.

The denominator is eligible matched recorded periods, not all goals records, all days or all participants. The 24-hour sensitivity check still does not prove complete wear. Missingness may be selective, so this rate need not describe records with missing floors. A chart makes the denominator visible rather than treating missing floor counts as zero.

In [ ]:
floors_results=[]
if 'activity' in joins:
    j=joins['activity']
    matched=j._merge.eq('both')
    context=j.sdv_group_id_goals.eq(j.sdv_group_id_other) & j.duration_measure_period_sec_goals.eq(j.duration_measure_period_sec_other)
    base=matched & context
    floor=j.floors_climbed
    goal=j.floors_goal
    eligible=base & goal.notna() & goal.gt(0) & floor.notna() & floor.ge(0)
    display(pd.DataFrame([{'goals_records':len(g),'matched_context_agrees':int(base.sum()),
      'matched_context_disagrees':int((matched & ~context).sum()),
      'missing_floor_count':int((base & floor.isna()).sum()),'recorded_zero_floors':int((base & floor.eq(0)).sum()),
      'negative_floors':int((base & floor.lt(0)).sum()),'missing_floors_goal':int((base & goal.isna()).sum()),
      'zero_floors_goal':int((base & goal.eq(0)).sum()),'negative_floors_goal':int((base & goal.lt(0)).sum())}]))
    for label,scope in [('all matched periods',base),('reported duration exactly 24 hours',base & j.duration_measure_period_sec_goals.eq(86400))]:
        use=scope & eligible
        met=use & floor.ge(goal)
        floors_results.append({'scope':label,'matched_records_in_scope':int(scope.sum()),'eligible_records':int(use.sum()),
          'excluded_records':int((scope & ~eligible).sum()),'records_meeting_goal':int(met.sum()),
          'percent_of_eligible_meeting_goal':100*met.sum()/use.sum() if use.any() else np.nan})
    display(pd.DataFrame(floors_results))
    counts=pd.Series({'Eligible: goal met':int((eligible & floor.ge(goal)).sum()),
        'Eligible: below goal':int((eligible & floor.lt(goal)).sum()),'Matched: excluded':int((base & ~eligible).sum())})
    counts.plot.barh(title='Floors: recorded-period denominator',xlabel='Records',figsize=(8,3))
    plt.tight_layout(); plt.show()


## Verified findings from this execution
- **987 records, 8 participants, 13 goals variables**; no missing goals-table values, exact duplicate rows or duplicated/null tested keys. All three targets are positive in this snapshot.
- Steps targets have **297 distinct values**, median **8,000**, range **2,780–30,000**. Floors targets are **10 (865 records)** or **20 (122 records)**. Intensity values are **9,000 (807)**, **12,000 (122)** and **18,000 (58)**; units and scope remain unconfirmed.
- Across **979 comparable adjacent observed-record transitions**, steps targets change **381 times** (178 increases, 203 decreases), involving **3 participants**. Two changes span date gaps. Floors and intensity targets show **no within-participant changes**; their overall variation is between participants in this snapshot.
- Exact-period joins match **all 987 records** to each comparison table, without unmatched periods. Compared contextual fields agree, and steps targets agree with the steps table.
- Floors are missing in **558 / 987 records (56.53%)**. There are **76 recorded zeros**, which are retained. With strictly positive goals and nonmissing, nonnegative floors, **98 / 429 eligible records (22.84%)** meet the target. The exactly-24-hour subset gives **98 / 427 (22.95%)**. These rates cannot describe the missing-floor records.
- **970** reporting periods are exactly 24 hours and **17** are not. All durations equal end minus start. Week labels disagree with ISO calendar-date conventions in **1 week-number record** and **9 weekday records**; investigate source conventions rather than silently replacing them. The steps notebook separately documents period overlap and timezone concerns.

**Takeaway:** Recorded steps targets vary over time for some participants, while floors and intensity settings remain constant within participants. Floors achievement is observable for fewer than half the records, so the eligible denominator must accompany its percentage. Intensity achievement is intentionally left uncomputed until semantics are confirmed.


## 7. Teacher-facing takeaway and limits
“I first checked what a goals row represents, then checked keys and measurement periods. I described all three targets and their changes within participants. I matched activity using the full recorded-period key and calculated floors achievement only for positive goals with nonmissing, nonnegative floors.”

Read the verified results above with their denominators. Target distributions are settings in this dataset, not recommendations. More observations give some participants more influence. Observed changes do not establish why a target changed. Zero targets need a source definition; missing floors cannot be recoded as inactivity. Window duration cannot prove full wear or target scope. **Intensity units, weighting and daily/weekly scope remain unknown**, so no intensity-achievement calculation is made.

Ask the data owner about target semantics, timezone/endpoints, enrolment dates, floors sensor availability and zero/sentinel conventions before drawing broader conclusions.